# ASR Test Transcription Generator

Generate Tamil transcriptions for the 579 test audio files using the trained Whisper Small model.

**Output Format:** `<test_file_id> <recognized_text>` (space-separated, one per line)

In [1]:
import os
import torch
import librosa
from tqdm.notebook import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor

In [2]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
WHISPER_MODEL = "vasista22/whisper-tamil-small"
CHECKPOINT_PATH = "train_data/checkpoints/best_model.pth"
TEST_DIR = "../../Final Dataset/Test"
OUTPUT_PATH = "../../Final Results/TEAM_NAME_Recognition_Run1.txt"

SAMPLING_RATE = 16000

Using device: cuda


In [3]:
# Load model and processor
print(f"Loading model from: {WHISPER_MODEL}")
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)

# Load fine-tuned checkpoint
print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
print(f"Best WER: {checkpoint.get('best_metric', 'unknown'):.2f}%")
print("Model loaded successfully!")

Loading model from: vasista22/whisper-tamil-small
Loading checkpoint from: train_data/checkpoints/best_model.pth
Loaded checkpoint from epoch 7
Best WER: 161.03%
Model loaded successfully!


In [4]:
# Get all test files
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])
print(f"Found {len(test_files)} test files")

Found 579 test files


In [5]:
# Process each file one by one with greedy decoding
results = []

with torch.no_grad():
    for filename in tqdm(test_files, desc="Generating transcriptions"):
        audio_path = os.path.join(TEST_DIR, filename)
        audio_id = os.path.splitext(filename)[0]  # e.g., test_0001
        
        try:
            # Load audio
            waveform, _ = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
            
            # Extract features
            inputs = processor.feature_extractor(
                waveform,
                sampling_rate=SAMPLING_RATE,
                return_tensors='pt',
                padding='max_length'
            )
            input_features = inputs['input_features'].to(DEVICE)
            
            # Generate with greedy decoding (temperature=0) and repetition penalty
            generated_ids = model.generate(
                input_features=input_features,
                temperature=0,  # Greedy decoding
                repetition_penalty=1.5,
                no_repeat_ngram_size=3,
            )
            
            # Decode transcription
            transcription = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            transcription = transcription.strip()
            
            results.append(f"{audio_id} {transcription}")
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            results.append(f"{audio_id} ")

print(f"\nProcessed {len(results)} files")

Generating transcriptions:   0%|          | 0/579 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Processed 579 files


In [6]:
# Save results in submission format
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(results))

print(f"Results saved to: {OUTPUT_PATH}")

Results saved to: ../../Final Results/TEAM_NAME_Recognition_Run1.txt


In [7]:
# Preview first 10 results
print("Preview (first 10 lines):")
for line in results[:10]:
    print(line)

Preview (first 10 lines):
test_0001 உம்ம். சொல்லவா பேரு ஏன் பேசிக் கிடப்பட்ட. வீடு திரு. இதை வெளங்குட மணியில், பையன் ஒரு பைன் ஏத்து படிகோறா பாலமுருகன் அவர்.
test_0002 வீட்டுக்காரங்க கடைக்கவே போறாங்கள். நான் வீடு சுத்துவேன், தமிழ.
test_0003 அதெல்லாம் நான் தான். பாக்குறேன். ஆமாம்.
test_0004 சொந்தகாரங்க அண்ணனிடமெல்லாம் பக்கத்துல தான இருக்கவு. எண் வீடுபே க்காலா எல்த் அதனால சொந்.க்ள சொவத்தை மூஞ்சி இருற்க.வெலயூர் ல இருவரு ஆவு அவங்களுக ௯ழ்ந்ருக .
test_0005 நான் சேரி அவங்க இது பங்சன். வெச்சா என்னையங்கள கூப்டுறது கண்டி வருவாங்ு. இல்ல எதும் பங்,அவங�வுங்கவோ போன்படி சொல்றீங்ளுவங� நாஙளும். போவோம்.
test_0006 ஏம்மாவு செய்றது அவரு ஏன் பையனுக்கு பிடிச்சது ப்ரோயாணி தான், சிக் கன்் பிரோ�ாணீ தா. அதனால வாரத்துல ஒரு நாள எப்படியாது அங்க செஞ்சு கொடுப்போம்.
test_0007 ஆம் ஒரு நாளைக்கு அவங்களுக்கவே டைம் கிடச்சன செய்வாங்க.
test_0008 எது பிடிக்கும்னா சாம்பார் தான் அப்படிகயெம்மாவே சாங்பருவ அவியல் வச்சா பிழைக்கவு.
test_0009 நான்குஜ்ல மீன் வப்போம். அப்றம் சிக்கன் தானடி கழிவாக்கவேண்டாம் மட்டை நீ எப்படி மாது ஒருதரவு.
t